# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StickRift/Rift/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My lane as an ML task

My lane is **scoring**. The goal is to give each content page a score representing how strongly it should be considered for a refresh review. This fits scoring because the output is a continuous priority signal rather than a fixed yes/no class. The score can combine multiple measured signals such as search volume, competition, content age, engagement, and recent performance. The output is decision-support for prioritizing which pages a content or SEO reviewer should examine first.


In [5]:
task_type = "scoring"
print("ML task type:", task_type)

ML task type: scoring


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or proxy

The target would be a **defined refresh-priority proxy**, rather than an observed outcome. The starter data does not contain a measured column showing whether a page was actually refreshed and what happened afterward. I would therefore define a proxy from measured signals that indicate a page may deserve review, such as recent performance, content age, engagement, and search opportunity. This proxy would be used to train or evaluate a prioritization score, while recognizing that it is a decision-support label rather than proof that a refresh will improve performance.


In [7]:
import pandas as pd

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

candidate_columns = [
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "engagement_rate",
    "search_volume",
    "competition",
    "cpc"
]

available_columns = [col for col in candidate_columns if col in df.columns]

print("Candidate proxy columns:")
print(available_columns)

Candidate proxy columns:
['content_age_days', 'impressions_90d', 'clicks_90d', 'engagement_rate', 'search_volume', 'competition', 'cpc']


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric

I would use **Precision@K** as the success metric, where K is the number of pages a reviewer can realistically examine. This measures the proportion of the top K scored pages that match the defined refresh-priority proxy. A higher Precision@K would mean the score is putting more relevant pages near the top of the review queue. I would choose K based on the review capacity available to the content team rather than treating one arbitrary cutoff as universally good.


In [9]:
k = 10
relevant_pages_in_top_k = 7

precision_at_k = relevant_pages_in_top_k / k

print(f"Example Precision@{k}: {precision_at_k:.2f}")

Example Precision@10: 0.70


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### The unit of analysis

The unit of analysis is **one content page**. Each row represents one pseudonymized content item with measured search, performance, engagement, and content characteristics. The scoring output would assign a refresh-priority score to each page so that pages can be compared and ranked for review. The dataset contains multiple measured signals for each page across the 90-day period and shorter recent periods.


In [11]:
# Show the unit of analysis as a real dataframe.
# One row = one content page.

lane_columns = [
    "content_type",
    "main_intent",
    "word_count",
    "content_age_days",
    "search_volume",
    "competition",
    "impressions_90d",
    "clicks_90d",
    "engagement_rate",
    "scroll_rate"
]

lane_slice = df[lane_columns].copy()

print("Rows:", lane_slice.shape[0])
print("Columns:", lane_slice.shape[1])
print("Unit of analysis: one row = one content page")

lane_slice.head(10)

Rows: 30000
Columns: 10
Unit of analysis: one row = one content page


,content_type,main_intent,word_count,content_age_days,search_volume,competition,impressions_90d,clicks_90d,engagement_rate,scroll_rate
0,keyword article,transactional,3221.0,187,10.0,0.67,3803,29,5.88,4.55
1,keyword article,informational,2481.0,445,90.0,0.01,15320,7,0.00,10.00
2,keyword article,informational,3515.0,141,0.0,0.00,12581,11,0.00,28.57
3,keyword article,commercial,NaN,463,10.0,0.00,11751,58,1.28,3.45
4,keyword article,informational,2803.0,263,0.0,0.00,19140,24,0.00,24.29
5,keyword article,transactional,3080.0,147,720.0,1.00,3970,1,0.00,25.00
6,keyword article,informational,3059.0,90,0.0,0.00,20,0,0.00,0.00
7,keyword article,commercial,NaN,445,590.0,0.44,1724,1,3.57,7.14
8,keyword article,informational,3807.0,90,0.0,0.00,32574,29,5.88,6.25
9,keyword article,informational,NaN,257,0.0,0.00,1240,2,0.00,0.00


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML beats a fixed rule here

A fixed rule could say something simple such as “review pages older than a certain number of days” or “review pages with a large decline in impressions.” However, the measured signals vary across pages. A page can have high search opportunity but low engagement, or strong impressions but an older content age. A single threshold would not capture these combinations well.

A scoring model can learn how multiple measured signals relate to the defined refresh-priority proxy and produce a more flexible ranking. This does not prove that ML will improve refresh outcomes, so the score should remain decision-support for a human reviewer. The value to test is whether the model produces a more useful review queue than a simple fixed rule.


In [13]:
# Check how much the measured signals vary across content pages.

variation_check = lane_slice[
    [
        "content_age_days",
        "search_volume",
        "competition",
        "impressions_90d",
        "clicks_90d",
        "engagement_rate"
    ]
].describe().T[
    ["min", "50%", "max"]
]

variation_check

,min,50%,max
content_age_days,90.0,236.0,564.0
search_volume,0.0,10.0,74000.0
competition,0.0,0.0,1.0
impressions_90d,1.0,731.0,517715.0
clicks_90d,0.0,1.0,4178.0
engagement_rate,0.0,0.0,100.0


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.